In [1]:
import shutil, os
import numpy as np
import pandas as pd
from sklearn.metrics import adjusted_rand_score
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
from results_processing import process_cluster_report, process_cluster_tsv, assign_nested_clusters

ddi_path = "/Volumes/imb-luckgr/projects/interface_clustering/results/benchmarking/benchmark_cluster/"
dmi_path = "/Volumes/imb-luckgr/projects/interface_clustering/results/benchmarking/benchmark_cluster/"

## Define functions

In [2]:
def generate_network_file(filepath, annotations, outpath):
    report = process_cluster_report(filepath)
    merge = pd.merge(report, annotations[['fullID','trueClusters','pairID','ClusterID']], left_on='query', right_on='fullID', how='left').drop('fullID',axis=1)
    merge.to_csv(outpath, sep="\t")

def add_ground_truth(df, annotations):
    merge = pd.merge(df, annotations[['fullID','trueClusters','pairID','ClusterID']], left_on='query', right_on='fullID', how='left').drop('fullID',axis=1)
    merge.rename(columns={'trueClusters':'if1clust','pairID':'pfamID1','ClusterID':'pfamclust1'}, inplace=True)
    merge2 = pd.merge(merge, annotations[['fullID','trueClusters','pairID','ClusterID']], left_on='target', right_on='fullID', how='left').drop('fullID',axis=1)
    merge2.rename(columns={'trueClusters':'if2clust','pairID':'pfamID2','ClusterID':'pfamclust2'}, inplace=True)

    # Create agreement label to distinguish 'similar' from 'dissimilar' interfaces based on cluster assignment
    merge2.loc[merge2['if1clust'] == merge2['if2clust'], 'agreement'] = 'Same cluster'
    merge2['agreement'] = merge2['agreement'].fillna('Different cluster')

    return merge2

In [68]:
def assess_benchmark(filepath1, filepath2, annotations):
    
    tsv1 = process_cluster_tsv(filepath1, dimer=True)
    tsv2 = process_cluster_tsv(filepath2, dimer=True)

    tsv2["predClust"] = pd.factorize(tsv2.representative)[0]

    assign_nested_clusters(tsv1, tsv2, clust_var="predClust")

    tsv1["trueClust"] = tsv1["member"].map(dict(zip(annotations.fullID, annotations.trueClusters)))
    tsv1.dropna(subset="trueClust", axis=0, inplace=True)
    
    print("Number of representatives from cluster TSV #1: ", tsv2.shape[0], 
          "\nNumber of predicted clusters: ", max(tsv1.parent_clust)+1, 
          "\nNumber of ground truth clusters: ", max(tsv1.trueClust)+1,
          "\nARS of predicted clusters compared to ground truth clusters: ", round(float(adjusted_rand_score(tsv1.trueClust, tsv1.parent_clust)),3))
    return tsv1, tsv2

In [52]:
def assess_benchmark_notnested(filepath1, annotations):
    
    tsv1 = process_cluster_tsv(filepath1, dimer=True)

    tsv1["predClust"] = pd.factorize(tsv1.representative)[0]

    tsv1["trueClust"] = tsv1["member"].map(dict(zip(annotations.fullID, annotations.trueClusters)))
    tsv1.dropna(subset="trueClust", axis=0, inplace=True)
    
    print("\nNumber of predicted clusters: ", max(tsv1.predClust)+1, 
          "\nNumber of ground truth clusters: ", max(tsv1.trueClust)+1,
          "\nARS of predicted clusters compared to ground truth clusters: ", round(float(adjusted_rand_score(tsv1.trueClust, tsv1.predClust)),3))
    return tsv1

In [4]:
# Read in ground truth cluster information
ddi_clusters = pd.read_csv('/Volumes/imb-luckgr/projects/interface_clustering/datasets/benchmarking/ddi_clusters.csv')
dmi_clusters = pd.read_csv('/Volumes/imb-luckgr/projects/interface_clustering/datasets/benchmarking/dmi_clusters.csv')

## DDI Analysis

In [23]:
ddi_path = "/Volumes/imb-luckgr/projects/interface_clustering/results/benchmarking/benchmark_cluster/"
# Create new folder containing only the representative members of the results of full-chain clustering
ddi_fc_clust = process_cluster_tsv(ddi_path+"ddidimer_clu_cluster.tsv", dimer=True)
ddi_fc_reps = list(set(ddi_fc_clust.representative))

In [69]:
ddi_nested_clust = assess_benchmark(filepath1 = ddi_path+"ddidimer_cluster.tsv",
                       filepath2 = ddi_path+"ddidimerrepint_0.3_cluster.tsv",
                       annotations = ddi_clusters)

Number of representatives from cluster TSV #1:  129 
Number of predicted clusters:  54 
Number of ground truth clusters:  41.0 
ARS of predicted clusters compared to ground truth clusters:  0.96


In [70]:
ddi_nested_clust = assess_benchmark(filepath1 = ddi_path+"ddidimer_cluster.tsv",
                       filepath2 = ddi_path+"ddidimerrepint_0.4_cluster.tsv",
                       annotations = ddi_clusters)

Number of representatives from cluster TSV #1:  129 
Number of predicted clusters:  59 
Number of ground truth clusters:  41.0 
ARS of predicted clusters compared to ground truth clusters:  0.944


In [72]:
ddi_nested_clust = assess_benchmark(filepath1 = ddi_path+"ddidimer_cluster.tsv",
                       filepath2 = ddi_path+"ddidimerrepint_0.5_cluster.tsv",
                       annotations = ddi_clusters)

Number of representatives from cluster TSV #1:  129 
Number of predicted clusters:  72 
Number of ground truth clusters:  41.0 
ARS of predicted clusters compared to ground truth clusters:  0.91


## DMI Analysis

In [73]:
dmi_path = "/Volumes/imb-luckgr/projects/interface_clustering/results/benchmarking/benchmark_cluster/"
# Create new folder containing only the representative members of the results of full-chain clustering
dmi_fc_clust = process_cluster_tsv(dmi_path+"dmidimer_clu_cluster.tsv", dimer=True)
dmi_fc_reps = list(set(dmi_fc_clust.representative))

In [74]:
dmi_nested_clust = assess_benchmark(filepath1 = dmi_path+"dmidimer_cluster.tsv",
                       filepath2 = dmi_path+"dmidimerrepint_0.5_cluster.tsv",
                       annotations = dmi_clusters)

Number of representatives from cluster TSV #1:  350 
Number of predicted clusters:  147 
Number of ground truth clusters:  58 
ARS of predicted clusters compared to ground truth clusters:  0.813


In [75]:
dmi_nested_clust = assess_benchmark(filepath1 = dmi_path+"dmidimer_cluster.tsv",
                       filepath2 = dmi_path+"dmidimerrepint_0.4_cluster.tsv",
                       annotations = dmi_clusters)

Number of representatives from cluster TSV #1:  350 
Number of predicted clusters:  116 
Number of ground truth clusters:  58 
ARS of predicted clusters compared to ground truth clusters:  0.895


In [76]:
dmi_nested_clust = assess_benchmark(filepath1 = dmi_path+"dmidimer_cluster.tsv",
                       filepath2 = dmi_path+"dmidimerrepint_0.3_cluster.tsv",
                       annotations = dmi_clusters)

Number of representatives from cluster TSV #1:  350 
Number of predicted clusters:  88 
Number of ground truth clusters:  58 
ARS of predicted clusters compared to ground truth clusters:  0.905
